# P300 Speller 实验分析

本 notebook 用于完成 P300 Speller 数据集的完整实验流程：数据读取、事件解析、标签构造、信号预处理、epoch 切分、特征提取、模型训练、交叉验证、字符解码、测试集预测以及结果可视化。

核心思路：先训练二分类模型判断每次行/列闪烁后是否出现 P300，再将同一字符试次内 5 轮刺激的 P300 得分按行列聚合，选出得分最高的行和列，最终映射为目标字符。

## 1. 环境与参数配置

如果 notebook 放在 `sy2` 目录下，默认会从当前目录的 `P300-S1` 文件夹读取数据。

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy import signal
from sklearn.base import clone
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# 路径配置
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "P300-S1"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_DATA_FILE = DATA_DIR / "S1_train_data.xlsx"
TRAIN_EVENT_FILE = DATA_DIR / "S1_train_event.xlsx"
TEST_DATA_FILE = DATA_DIR / "S1_test_data.xlsx"
TEST_EVENT_FILE = DATA_DIR / "S1_test_event.xlsx"

# 信号参数
FS = 250
BANDPASS = (0.5, 35.0)
USE_NOTCH = True
NOTCH_HZ = 50.0

# epoch 参数：P300 通常在刺激后约 300 ms 附近，但这里保留更完整窗口
EPOCH_TMIN = -0.100
EPOCH_TMAX = 0.800
BASELINE = (-0.100, 0.000)
FEATURE_WINDOW = (0.000, 0.800)

# 特征降采样。250 Hz / 5 = 50 Hz，可明显降低特征维度。
DOWNSAMPLE_FACTOR = 5

# event 表第二列是采样点编号。若按 Excel/Matlab 风格从 1 开始编号，则需要减 1。
# 如后续发现 epoch 整体错位，可把 SAMPLE_OFFSET 改成 0 后重跑。
SAMPLE_OFFSET = 1

RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("DATA_DIR:", DATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. 字符矩阵与编码函数

In [ ]:
CHAR_MATRIX = np.array([
    list("ABCDEF"),
    list("GHIJKL"),
    list("MNOPQR"),
    list("STUVWX"),
    list("YZ1234"),
    list("56789_"),
])

def target_code_to_char(target_code: int) -> str:
    """训练集首行目标编号转字符。101=A, 102=B, ..., 136=_."""
    idx = int(target_code) - 100
    if not 1 <= idx <= 36:
        return "?"
    row = (idx - 1) // 6
    col = (idx - 1) % 6
    return CHAR_MATRIX[row, col]

def target_code_to_row_col(target_code: int) -> tuple[int, int]:
    """返回 1-based 目标行、目标列。"""
    idx = int(target_code) - 100
    if not 1 <= idx <= 36:
        raise ValueError(f"非法目标编码: {target_code}")
    row = math.ceil(idx / 6)
    col = ((idx - 1) % 6) + 1
    return row, col

def target_code_to_event_codes(target_code: int) -> tuple[int, int]:
    """返回目标行刺激编号和目标列刺激编号。行为 1-6，列为 7-12。"""
    row, col = target_code_to_row_col(target_code)
    row_event_code = row
    col_event_code = col + 6
    return row_event_code, col_event_code

def row_col_to_char(row: int, col: int) -> str:
    """1-based 行列转字符。"""
    return CHAR_MATRIX[row - 1, col - 1]

def decode_scores_to_char(code_scores: dict[int, float]) -> dict:
    """根据 1-12 行列刺激得分解码字符。"""
    row_code = max(range(1, 7), key=lambda c: code_scores.get(c, -np.inf))
    col_code = max(range(7, 13), key=lambda c: code_scores.get(c, -np.inf))
    row = row_code
    col = col_code - 6
    return {
        "row_event": row_code,
        "col_event": col_code,
        "row": row,
        "col": col,
        "char": row_col_to_char(row, col),
    }

display(pd.DataFrame(CHAR_MATRIX, index=[f"行{i}" for i in range(1, 7)], columns=[f"列{i}" for i in range(1, 7)]))

## 3. 读取 Excel 与数据结构检查

注意：所有 Excel 都必须使用 `header=None` 读取，否则第一行数据会被误认为表头。

In [ ]:
def read_workbook_sheets(path: Path) -> dict[str, pd.DataFrame]:
    """读取一个 xlsx 文件中的非空 sheet，统一 header=None。"""
    xl = pd.ExcelFile(path)
    sheets = {}
    for sheet_name in xl.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet_name, header=None)
        if df.empty:
            continue
        sheets[sheet_name] = df
    return sheets

def summarize_workbooks(data_sheets: dict[str, pd.DataFrame], event_sheets: dict[str, pd.DataFrame], known_targets: bool) -> pd.DataFrame:
    rows = []
    for sheet_name, data_df in data_sheets.items():
        event_df = event_sheets.get(sheet_name)
        target_code = None
        target_char = None
        n_events = None
        n_valid_stim = None
        n_round_end = None
        if event_df is not None and not event_df.empty:
            target_code = int(event_df.iloc[0, 0])
            target_char = target_code_to_char(target_code) if known_targets else "unknown"
            event_codes = pd.to_numeric(event_df.iloc[1:, 0], errors="coerce").dropna().astype(int)
            n_events = len(event_codes)
            n_valid_stim = int(event_codes.between(1, 12).sum())
            n_round_end = int((event_codes == 100).sum())
        rows.append({
            "sheet": sheet_name,
            "target_code": target_code,
            "target_char": target_char,
            "samples": data_df.shape[0],
            "channels": data_df.shape[1],
            "event_rows_after_header": n_events,
            "valid_stim_1_12": n_valid_stim,
            "round_end_100": n_round_end,
        })
    return pd.DataFrame(rows)

train_data_sheets = read_workbook_sheets(TRAIN_DATA_FILE)
train_event_sheets = read_workbook_sheets(TRAIN_EVENT_FILE)
test_data_sheets = read_workbook_sheets(TEST_DATA_FILE)
test_event_sheets = read_workbook_sheets(TEST_EVENT_FILE)

train_summary = summarize_workbooks(train_data_sheets, train_event_sheets, known_targets=True)
test_summary = summarize_workbooks(test_data_sheets, test_event_sheets, known_targets=False)

print("训练集概览")
display(train_summary)
print("测试集概览")
display(test_summary)

In [ ]:
# 可视化一个训练试次的事件序列，检查每轮 12 个刺激 + 100 结束标记。
example_sheet = next(iter(train_event_sheets.keys()))
event_df = train_event_sheets[example_sheet]
events_plot = event_df.iloc[1:].copy()
events_plot.columns = ["event_code", "sample"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(events_plot["sample"], events_plot["event_code"], c=np.where(events_plot["event_code"] == 100, "tab:red", "tab:blue"))
ax.set_title(f"事件序列示例：{example_sheet}")
ax.set_xlabel("采样点")
ax.set_ylabel("事件编号")
ax.set_yticks(list(range(1, 13)) + [100])
plt.show()

display(events_plot.head(20))

## 4. 信号预处理与 Epoch 切分函数

In [ ]:
def preprocess_eeg(raw: np.ndarray, fs: int = FS) -> np.ndarray:
    """对连续 EEG 进行带通滤波和可选陷波。输入形状为 samples x channels。"""
    x = np.asarray(raw, dtype=float)
    x = x - np.nanmean(x, axis=0, keepdims=True)
    x = np.nan_to_num(x, nan=0.0)

    low, high = BANDPASS
    sos = signal.butter(4, [low, high], btype="bandpass", fs=fs, output="sos")
    x = signal.sosfiltfilt(sos, x, axis=0)

    if USE_NOTCH and NOTCH_HZ < fs / 2:
        b, a = signal.iirnotch(w0=NOTCH_HZ, Q=30, fs=fs)
        x = signal.filtfilt(b, a, x, axis=0)
    return x

def epoch_time_vector(fs: int = FS) -> np.ndarray:
    start = int(round(EPOCH_TMIN * fs))
    stop = int(round(EPOCH_TMAX * fs))
    return np.arange(start, stop) / fs

TIMES = epoch_time_vector()

def extract_epoch(preprocessed: np.ndarray, event_sample: int, fs: int = FS) -> np.ndarray | None:
    """按事件采样点提取 epoch。返回 channels x time。越界则返回 None。"""
    sample_idx = int(event_sample) - SAMPLE_OFFSET
    start_offset = int(round(EPOCH_TMIN * fs))
    stop_offset = int(round(EPOCH_TMAX * fs))
    start = sample_idx + start_offset
    stop = sample_idx + stop_offset
    if start < 0 or stop > preprocessed.shape[0] or stop <= start:
        return None

    epoch = preprocessed[start:stop, :].T.copy()

    # 基线校正：按 epoch 时间向量选取 -100ms 到 0ms。
    b0, b1 = BASELINE
    baseline_mask = (TIMES >= b0) & (TIMES < b1)
    if baseline_mask.any():
        epoch = epoch - epoch[:, baseline_mask].mean(axis=1, keepdims=True)
    return epoch

def parse_event_rows(event_df: pd.DataFrame) -> pd.DataFrame:
    """解析 event sheet，返回去掉首行后的事件表。"""
    rows = event_df.iloc[1:, :2].copy()
    rows.columns = ["event_code", "sample"]
    rows["event_code"] = pd.to_numeric(rows["event_code"], errors="coerce")
    rows["sample"] = pd.to_numeric(rows["sample"], errors="coerce")
    rows = rows.dropna().astype({"event_code": int, "sample": int})
    return rows

def build_epochs_for_sheet(data_df: pd.DataFrame, event_df: pd.DataFrame, sheet_name: str, known_target: bool) -> tuple[list[np.ndarray], list[int | None], list[dict]]:
    """对单个字符试次构造 epochs、标签和元数据。"""
    raw = data_df.to_numpy(dtype=float)
    filtered = preprocess_eeg(raw)
    target_code = int(event_df.iloc[0, 0])

    if known_target:
        row_event, col_event = target_code_to_event_codes(target_code)
        target_events = {row_event, col_event}
        target_char = target_code_to_char(target_code)
    else:
        row_event, col_event = None, None
        target_events = set()
        target_char = "unknown"

    event_rows = parse_event_rows(event_df)
    epochs, labels, metas = [], [], []
    skipped = 0

    for event_index, row in event_rows.iterrows():
        event_code = int(row["event_code"])
        event_sample = int(row["sample"])
        if event_code == 100:
            continue
        if not 1 <= event_code <= 12:
            continue

        epoch = extract_epoch(filtered, event_sample)
        if epoch is None:
            skipped += 1
            continue

        label = int(event_code in target_events) if known_target else None
        epochs.append(epoch)
        labels.append(label)
        metas.append({
            "sheet": sheet_name,
            "event_index": int(event_index),
            "event_code": event_code,
            "event_sample": event_sample,
            "target_code": target_code,
            "target_char": target_char,
            "target_row_event": row_event,
            "target_col_event": col_event,
            "skipped_in_sheet": skipped,
        })

    return epochs, labels, metas

def build_dataset(data_sheets: dict[str, pd.DataFrame], event_sheets: dict[str, pd.DataFrame], known_target: bool) -> tuple[np.ndarray, np.ndarray | None, pd.DataFrame]:
    all_epochs, all_labels, all_metas = [], [], []
    for sheet_name, data_df in data_sheets.items():
        if sheet_name not in event_sheets:
            print(f"跳过 {sheet_name}: 找不到对应 event sheet")
            continue
        epochs, labels, metas = build_epochs_for_sheet(data_df, event_sheets[sheet_name], sheet_name, known_target=known_target)
        all_epochs.extend(epochs)
        all_labels.extend(labels)
        all_metas.extend(metas)

    X_epochs = np.stack(all_epochs, axis=0)
    meta_df = pd.DataFrame(all_metas)
    if known_target:
        y = np.asarray(all_labels, dtype=int)
    else:
        y = None
    return X_epochs, y, meta_df

def epochs_to_features(epochs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """将 n x channels x time 的 epoch 转为模型特征。"""
    feature_mask = (TIMES >= FEATURE_WINDOW[0]) & (TIMES < FEATURE_WINDOW[1])
    selected = epochs[:, :, feature_mask]
    selected_times = TIMES[feature_mask]
    if DOWNSAMPLE_FACTOR and DOWNSAMPLE_FACTOR > 1:
        selected = selected[:, :, ::DOWNSAMPLE_FACTOR]
        selected_times = selected_times[::DOWNSAMPLE_FACTOR]
    features = selected.reshape(selected.shape[0], -1)
    return features, selected_times

## 5. 构造训练样本

In [ ]:
train_epochs, y, train_meta = build_dataset(train_data_sheets, train_event_sheets, known_target=True)
X, feature_times = epochs_to_features(train_epochs)

print("epochs shape:", train_epochs.shape, "= 样本数 x 通道数 x 时间点")
print("features shape:", X.shape)
print("labels shape:", y.shape)
display(train_meta.head())

class_counts = pd.Series(y).map({0: "无 P300", 1: "有 P300"}).value_counts()
display(class_counts.rename("样本数").to_frame())

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind="bar", ax=ax, color=["tab:gray", "tab:orange"])
ax.set_title("训练样本类别分布")
ax.set_ylabel("epoch 数")
ax.tick_params(axis="x", rotation=0)
plt.show()

## 6. ERP 可视化

这里先查看目标刺激与非目标刺激的 grand average ERP。由于没有通道名称和电极位置，先展示 20 通道平均波形和各通道差异热图。

In [ ]:
target_epochs = train_epochs[y == 1]
nontarget_epochs = train_epochs[y == 0]

target_mean = target_epochs.mean(axis=0)
nontarget_mean = nontarget_epochs.mean(axis=0)

target_global = target_mean.mean(axis=0)
nontarget_global = nontarget_mean.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(TIMES * 1000, target_global, label="目标刺激：有 P300", color="tab:orange")
ax.plot(TIMES * 1000, nontarget_global, label="非目标刺激：无 P300", color="tab:blue")
ax.axvspan(250, 700, color="tab:orange", alpha=0.12, label="P300 关注窗口")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("目标 vs 非目标 Grand Average ERP（20 通道平均）")
ax.set_xlabel("刺激后时间 (ms)")
ax.set_ylabel("相对电位")
ax.legend()
plt.show()

diff = target_mean - nontarget_mean
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(diff, aspect="auto", cmap="RdBu_r", extent=[TIMES[0] * 1000, TIMES[-1] * 1000, diff.shape[0], 1])
ax.axvline(0, color="black", linewidth=1)
ax.axvspan(250, 700, color="yellow", alpha=0.12)
ax.set_title("各通道目标-非目标 ERP 差异热图")
ax.set_xlabel("刺激后时间 (ms)")
ax.set_ylabel("通道编号")
fig.colorbar(im, ax=ax, label="目标 - 非目标")
plt.show()

## 7. 模型定义

使用两个稳健的线性模型：Shrinkage LDA 和带类别权重的 Logistic Regression。

In [ ]:
models = {
    "Shrinkage LDA": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)),
    ]),
}

def model_scores(model, X_part: np.ndarray) -> np.ndarray:
    """返回越大越像 P300 的连续得分。优先使用 P300 概率。"""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_part)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X_part)
    return model.predict(X_part).astype(float)

def aggregate_code_scores(meta_part: pd.DataFrame, scores: np.ndarray, agg: str = "mean") -> dict[int, float]:
    temp = meta_part[["event_code"]].copy()
    temp["score"] = scores
    if agg == "sum":
        series = temp.groupby("event_code")["score"].sum()
    else:
        series = temp.groupby("event_code")["score"].mean()
    return {int(k): float(v) for k, v in series.to_dict().items()}

def decode_trial(meta_part: pd.DataFrame, scores: np.ndarray, agg: str = "mean") -> dict:
    code_scores = aggregate_code_scores(meta_part, scores, agg=agg)
    decoded = decode_scores_to_char(code_scores)
    decoded["code_scores"] = code_scores
    return decoded

## 8. Leave-One-Character-Out 交叉验证

按字符试次分组交叉验证，避免同一字符内高度相关的 epoch 同时出现在训练集和验证集。

In [ ]:
def leave_one_character_out_cv(model, X: np.ndarray, y: np.ndarray, meta: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    groups = meta["sheet"].unique()
    fold_rows = []
    y_true_all, y_pred_all, score_all = [], [], []

    for sheet in groups:
        val_mask = (meta["sheet"] == sheet).to_numpy()
        train_mask = ~val_mask

        fitted = clone(model)
        fitted.fit(X[train_mask], y[train_mask])

        val_scores = model_scores(fitted, X[val_mask])
        val_pred = (val_scores >= 0.5).astype(int)

        y_val = y[val_mask]
        y_true_all.extend(y_val.tolist())
        y_pred_all.extend(val_pred.tolist())
        score_all.extend(val_scores.tolist())

        val_meta = meta.loc[val_mask].reset_index(drop=True)
        decoded = decode_trial(val_meta, val_scores, agg="mean")
        actual_char = val_meta["target_char"].iloc[0]

        fold_rows.append({
            "sheet": sheet,
            "actual_char": actual_char,
            "pred_char": decoded["char"],
            "char_correct": decoded["char"] == actual_char,
            "actual_row_event": int(val_meta["target_row_event"].iloc[0]),
            "actual_col_event": int(val_meta["target_col_event"].iloc[0]),
            "pred_row_event": decoded["row_event"],
            "pred_col_event": decoded["col_event"],
            "epoch_accuracy": accuracy_score(y_val, val_pred),
            "epoch_balanced_accuracy": balanced_accuracy_score(y_val, val_pred),
            "epoch_f1": f1_score(y_val, val_pred, zero_division=0),
            "epoch_auc": roc_auc_score(y_val, val_scores) if len(np.unique(y_val)) == 2 else np.nan,
        })

    y_true_all = np.asarray(y_true_all)
    y_pred_all = np.asarray(y_pred_all)
    score_all = np.asarray(score_all)

    overall = pd.DataFrame([{
        "epoch_accuracy": accuracy_score(y_true_all, y_pred_all),
        "epoch_balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "epoch_precision": precision_score(y_true_all, y_pred_all, zero_division=0),
        "epoch_recall": recall_score(y_true_all, y_pred_all, zero_division=0),
        "epoch_f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "epoch_auc": roc_auc_score(y_true_all, score_all),
        "character_accuracy": np.mean([r["char_correct"] for r in fold_rows]),
    }])

    return pd.DataFrame(fold_rows), overall, y_true_all, y_pred_all, score_all

cv_results = {}
overall_rows = []

for model_name, model in models.items():
    fold_df, overall_df, y_true_cv, y_pred_cv, score_cv = leave_one_character_out_cv(model, X, y, train_meta)
    cv_results[model_name] = {
        "folds": fold_df,
        "overall": overall_df,
        "y_true": y_true_cv,
        "y_pred": y_pred_cv,
        "scores": score_cv,
    }
    row = overall_df.iloc[0].to_dict()
    row["model"] = model_name
    overall_rows.append(row)

overall_cv = pd.DataFrame(overall_rows).set_index("model")
display(overall_cv)

for model_name, result in cv_results.items():
    print("\n", model_name)
    display(result["folds"][["sheet", "actual_char", "pred_char", "char_correct", "actual_row_event", "actual_col_event", "pred_row_event", "pred_col_event", "epoch_auc"]])

In [ ]:
# 交叉验证混淆矩阵
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4))
if len(models) == 1:
    axes = [axes]

for ax, (model_name, result) in zip(axes, cv_results.items()):
    cm = confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["无 P300", "有 P300"])
    disp.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(model_name)
plt.tight_layout()
plt.show()

# 字符级验证结果柱状图
char_acc = overall_cv["character_accuracy"].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
char_acc.plot(kind="bar", ax=ax, color="tab:green")
ax.set_ylim(0, 1.05)
ax.set_ylabel("字符识别准确率")
ax.set_title("Leave-One-Character-Out 字符级准确率")
ax.tick_params(axis="x", rotation=20)
plt.show()

## 9. 在完整训练集上训练最终模型并检查训练集解码

这里的训练集解码是对模型拟合同一训练集后的 sanity check，不能替代交叉验证结果。

In [ ]:
final_models = {}
train_decode_rows = []

for model_name, model in models.items():
    fitted = clone(model)
    fitted.fit(X, y)
    final_models[model_name] = fitted

    scores = model_scores(fitted, X)
    for sheet in train_meta["sheet"].unique():
        mask = (train_meta["sheet"] == sheet).to_numpy()
        meta_part = train_meta.loc[mask].reset_index(drop=True)
        decoded = decode_trial(meta_part, scores[mask], agg="mean")
        actual_char = meta_part["target_char"].iloc[0]
        train_decode_rows.append({
            "model": model_name,
            "sheet": sheet,
            "actual_char": actual_char,
            "pred_char": decoded["char"],
            "correct": decoded["char"] == actual_char,
            "pred_row_event": decoded["row_event"],
            "pred_col_event": decoded["col_event"],
        })

train_decode_df = pd.DataFrame(train_decode_rows)
display(train_decode_df)
display(train_decode_df.groupby("model")["correct"].mean().rename("训练集字符解码准确率").to_frame())

## 10. 行列得分可视化函数

In [ ]:
def plot_code_scores(code_scores: dict[int, float], title: str):
    codes = list(range(1, 13))
    values = [code_scores.get(c, np.nan) for c in codes]
    colors = ["tab:blue" if c <= 6 else "tab:orange" for c in codes]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(codes, values, color=colors)
    ax.set_xticks(codes)
    ax.set_xlabel("刺激编号：1-6 为行，7-12 为列")
    ax.set_ylabel("聚合 P300 得分")
    ax.set_title(title)
    plt.show()

# 展示一个训练试次的最终模型行列得分
demo_model_name = max(overall_cv.index, key=lambda name: overall_cv.loc[name, "character_accuracy"])
demo_model = final_models[demo_model_name]
demo_sheet = train_meta["sheet"].unique()[0]
demo_mask = (train_meta["sheet"] == demo_sheet).to_numpy()
demo_scores = model_scores(demo_model, X[demo_mask])
demo_decoded = decode_trial(train_meta.loc[demo_mask].reset_index(drop=True), demo_scores)
plot_code_scores(demo_decoded["code_scores"], f"训练试次 {demo_sheet} 行列 P300 得分：{demo_model_name}")

## 11. 测试集预测

测试集首行 `666` 是未知目标占位符，不参与标签构造。下面对每个测试 sheet 提取 epoch，使用最终模型预测每个刺激的 P300 得分，再解码字符。

In [ ]:
test_epochs, _, test_meta = build_dataset(test_data_sheets, test_event_sheets, known_target=False)
X_test, _ = epochs_to_features(test_epochs)

print("test epochs shape:", test_epochs.shape)
print("test features shape:", X_test.shape)
display(test_meta.head())

test_decode_rows = []
test_score_maps = {}

for model_name, fitted in final_models.items():
    scores = model_scores(fitted, X_test)
    chars = []
    for sheet in test_meta["sheet"].unique():
        mask = (test_meta["sheet"] == sheet).to_numpy()
        meta_part = test_meta.loc[mask].reset_index(drop=True)
        decoded = decode_trial(meta_part, scores[mask], agg="mean")
        chars.append(decoded["char"])
        test_score_maps[(model_name, sheet)] = decoded["code_scores"]
        test_decode_rows.append({
            "model": model_name,
            "sheet": sheet,
            "pred_char": decoded["char"],
            "pred_row_event": decoded["row_event"],
            "pred_col_event": decoded["col_event"],
            "pred_row": decoded["row"],
            "pred_col": decoded["col"],
        })
    print(f"{model_name} 测试集预测字符串:", "".join(chars))

test_decode_df = pd.DataFrame(test_decode_rows)
display(test_decode_df)

prediction_path = OUTPUT_DIR / "p300_test_predictions.csv"
test_decode_df.to_csv(prediction_path, index=False, encoding="utf-8-sig")
print("预测结果已保存:", prediction_path)

In [ ]:
# 展示每个测试 sheet 的行列得分。图较多，可按需要修改 selected_model。
selected_model = demo_model_name
for sheet in test_meta["sheet"].unique():
    code_scores = test_score_maps[(selected_model, sheet)]
    pred_char = test_decode_df[(test_decode_df["model"] == selected_model) & (test_decode_df["sheet"] == sheet)]["pred_char"].iloc[0]
    plot_code_scores(code_scores, f"测试试次 {sheet} 行列 P300 得分：{selected_model}，预测 {pred_char}")

## 12. 保存关键结果表

In [ ]:
overall_cv_path = OUTPUT_DIR / "p300_cv_overall.csv"
train_decode_path = OUTPUT_DIR / "p300_train_decode.csv"

overall_cv.to_csv(overall_cv_path, encoding="utf-8-sig")
train_decode_df.to_csv(train_decode_path, index=False, encoding="utf-8-sig")

for model_name, result in cv_results.items():
    safe_name = model_name.lower().replace(" ", "_")
    result["folds"].to_csv(OUTPUT_DIR / f"p300_cv_folds_{safe_name}.csv", index=False, encoding="utf-8-sig")

print("已保存：")
print(overall_cv_path)
print(train_decode_path)
print(prediction_path)

## 13. 可调整方向

如果交叉验证或测试预测不理想，优先尝试以下调整：

1. 将 `SAMPLE_OFFSET` 从 `1` 改为 `0`，检查事件采样点是否存在 1 个采样点偏移。
2. 调整 `FEATURE_WINDOW`，例如 `(0.250, 0.700)` 或 `(0.300, 0.700)`。
3. 调整 `BANDPASS`，例如 `(0.1, 30.0)` 或 `(0.5, 20.0)`。
4. 比较 `mean` 与 `sum` 两种行列得分聚合方式。
5. 添加 xDAWN 或 CSP 等空间滤波方法。
6. 如果已知通道名称和电极位置，可进一步做通道选择或头皮图分析。

## 14. 严格分组验证重测

以下验证以完整字符 sheet 为分组单位，外层验证集不参与候选配置选择，避免把同一字符试次的高度相关 epoch 随机拆到训练集和验证集。当前 12 个已知字符上的严格外层结果为字符准确率 50%、行准确率 75%、列准确率 75%。样本量很小，因此该结果只用于估计泛化能力，不能用于证明无标签测试集预测正确。

In [ ]:
# 运行严格外层分组验证；结果写入 outputs_validation_retest_fast/
from group_validation_retest_fast import main as run_group_validation_retest

run_group_validation_retest()

## 15. 最终集成模型与测试集预测

集成模型只利用训练集标签确定配置。测试集的目标字段为 666，占位符不能作为标签或调参依据。输出字符仅为模型预测；在没有测试集真实标签时，不能计算或保证测试准确率。

In [ ]:
# 使用训练集拟合三个互补模型，标准化各刺激得分后集成。
# 结果写入 outputs_best/。
from best_subset_predict import main as run_final_ensemble_prediction

run_final_ensemble_prediction()